# CU28 mixed_context - Data Sources Audit

Notebook narrativo de auditoria para el scope `mixed_context`.


## Objetivo

Revisar de forma explicita las fuentes activas, trazadas y candidatas, y verificar su trazabilidad local mediante manifests, URLs oficiales, licencias y artefactos derivados.


## Alcance

Este analisis describe la ruta oficial reproducible `mixed_context`. Las senales externas se tratan como contexto/proxy. Las variables internas de planta siguen siendo sinteticas salvo carga posterior de cliente.


## Inputs

            - `docs/data_sources_registry.md`
- `data/raw/external/*/source_manifest.json`
- `data/raw/external/raw_manifest__mixed_context.json`
- `data_blob_manifest.json`


## Outputs esperados

            - `reports/tables/eda/data_sources_audit__mixed_context.csv`
- `reports/tables/eda/data_sources_inconsistencies__mixed_context.csv`
- `reports/figures/eda/data_sources_status_counts__mixed_context.png`
- `reports/figures/eda/data_sources_type_counts__mixed_context.png`
- `reports/figures/eda/data_sources_raw_file_counts__mixed_context.png`


## Limitaciones

Este notebook documenta evidencia reproducible del pipeline oficial, pero no sustituye la revision de codigo, la auditoria de datos de origen ni una certificacion operacional de planta.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.reproducibility.notebook_support import (
    detect_temporal_columns,
    ensure_eda_dirs,
    execution_metadata,
    first_valid_temporal_range,
    load_source_manifests,
    load_tabular_file,
    parse_markdown_table,
    print_frame,
    print_series,
    project_root,
    read_json,
    relative_to_root,
    save_figure,
    save_table,
    sha256_file,
)


In [ ]:
NOTEBOOK_NAME = "00_data_sources_audit.ipynb"
PROJECT_ROOT = project_root()
SCOPE = globals().get("scope", "mixed_context")
REPORT_DIRS = ensure_eda_dirs()
META = execution_metadata(SCOPE)
FIGURES = []
TABLES = []
print(json.dumps(META, indent=2))


## Carga de datos

Las siguientes celdas cargan los artefactos de entrada y muestran verificaciones intermedias antes de producir tablas y graficas.


In [ ]:
registry_path = PROJECT_ROOT / "docs" / "data_sources_registry.md"
registry_table = parse_markdown_table(registry_path)
for column in registry_table.columns:
    registry_table[column] = registry_table[column].astype(str).str.replace("`", "", regex=False).str.strip()
print_frame("Registry markdown table", registry_table, rows=10)
display(registry_table.head(10))


In [ ]:
raw_root = PROJECT_ROOT / "data" / "raw" / "external"
source_manifests = load_source_manifests(raw_root)
manifest_df = pd.DataFrame(
    [
        {
            "source_id": item["source_id"],
            "organization": item["organization"],
            "source_type": item["source_type"],
            "evidence_status": item["evidence_status"],
            "official_url": item["official_url"],
            "download_url_or_endpoint": item["download_url_or_endpoint"],
            "license_or_terms_url": item["license_or_terms_url"],
            "access_date": item["access_date"],
            "retrieval_method": item["retrieval_method"],
            "role": item["role"],
            "limitations": item["limitations"],
            "raw_files_count": len(item.get("raw_files", [])),
            "derived_artifacts_count": len(item.get("derived_artifacts", [])),
        }
        for item in source_manifests
    ]
)
print_frame("Manifest summary", manifest_df, rows=10)
display(manifest_df)


## Inspeccion inicial

Se cruza la tabla de documentacion con los manifests reales para comprobar si ambas vistas coinciden en estado, URLs y rutas locales.


In [ ]:
source_overview = registry_table.merge(
    manifest_df,
    left_on="source_id",
    right_on="source_id",
    how="left",
    suffixes=("_registry", "_manifest"),
)
source_overview["status_match"] = source_overview["status"].eq(source_overview["evidence_status"])
print_frame(
    "Overview after joining registry and manifests",
    source_overview[["source_id", "status", "evidence_status", "source_type", "raw_files_count", "derived_artifacts_count", "status_match"]],
    rows=10,
)
display(source_overview[["source_id", "status", "evidence_status", "source_type", "status_match"]])


In [ ]:
url_checks = source_overview[[
    "source_id",
    "official_url_registry",
    "download_url_or_endpoint_registry",
    "license_or_terms_url_registry",
]].copy()
url_checks["official_url_ok"] = url_checks["official_url_registry"].astype(str).str.startswith("http")
url_checks["download_url_ok"] = url_checks["download_url_or_endpoint_registry"].astype(str).str.startswith("http")
url_checks["license_url_ok"] = url_checks["license_or_terms_url_registry"].astype(str).str.startswith("http")
print_frame("URL checks", url_checks, rows=10)
display(url_checks)


In [ ]:
source_to_manifest = {item["source_id"]: item for item in source_manifests}
existence_rows = []
for source_id, manifest in source_to_manifest.items():
    raw_paths = [entry["path"] for entry in manifest.get("raw_files", [])]
    derived_paths = manifest.get("derived_artifacts", [])
    existence_rows.append(
        {
            "source_id": source_id,
            "raw_exists": all((PROJECT_ROOT / raw_path).exists() for raw_path in raw_paths) if raw_paths else False,
            "processed_exists": all((PROJECT_ROOT / derived_path).exists() for derived_path in derived_paths) if derived_paths else False,
            "hash_available": all(bool(entry.get("sha256")) for entry in manifest.get("raw_files", [])),
            "raw_path_count": len(raw_paths),
            "derived_path_count": len(derived_paths),
        }
    )
existence_df = pd.DataFrame(existence_rows)
print_frame("Existence checks", existence_df, rows=10)
display(existence_df)


## Interpretacion intermedia

Las fuentes `active` deben tener URL oficial, licencia, raw local y artefactos procesados. La fuente `traced` puede quedar como evidencia de contexto siempre que no se defienda como feed semanal activo.


In [ ]:
inconsistencies_df = source_overview[[
    "source_id",
    "status",
    "evidence_status",
    "status_match",
]].merge(existence_df, on="source_id", how="left").merge(
    url_checks[["source_id", "official_url_ok", "download_url_ok", "license_url_ok"]],
    on="source_id",
    how="left",
)
inconsistencies_df["has_inconsistency"] = ~(
    inconsistencies_df["status_match"]
    & inconsistencies_df["official_url_ok"]
    & inconsistencies_df["license_url_ok"]
    & (
        (inconsistencies_df["status"] != "active")
        | (inconsistencies_df["raw_exists"] & inconsistencies_df["processed_exists"])
    )
)
print_frame("Inconsistency table", inconsistencies_df, rows=10)
display(inconsistencies_df)


In [ ]:
status_counts = source_overview["evidence_status"].fillna("missing_manifest").value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4))
status_counts.plot(kind="bar", color=["#2f6f4f", "#c98f2b", "#7f8c8d"], ax=ax)
ax.set_title("Sources by evidence status")
ax.set_xlabel("evidence_status")
ax.set_ylabel("source_count")
FIGURES.append(save_figure(fig, "data_sources_status_counts__mixed_context.png"))
plt.close(fig)
print(status_counts.to_string())


### Interpretacion de la figura

La barra de estados debe mostrar que solo INE_CPI y MAPA_SLAUGHTER_MAPA quedan activos en la ruta oficial. MAPA_PRICES_OM se conserva como trazado y no como serie semanal defendible.


In [ ]:
type_counts = manifest_df["source_type"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 4))
type_counts.plot(kind="bar", color="#3c7dc4", ax=ax)
ax.set_title("Sources by source_type")
ax.set_xlabel("source_type")
ax.set_ylabel("source_count")
FIGURES.append(save_figure(fig, "data_sources_type_counts__mixed_context.png"))
plt.close(fig)
print(type_counts.to_string())


### Interpretacion de la figura

La mezcla de CSV oficial y bundle de hojas de calculo confirma que el pipeline parte de snapshots heterogeneos. Por eso el manifest por fuente es necesario para una auditoria reproducible.


In [ ]:
raw_file_counts = manifest_df.set_index("source_id")["raw_files_count"].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
raw_file_counts.plot(kind="bar", color="#7a4b94", ax=ax)
ax.set_title("Raw files restored per source")
ax.set_xlabel("source_id")
ax.set_ylabel("raw_files_count")
FIGURES.append(save_figure(fig, "data_sources_raw_file_counts__mixed_context.png"))
plt.close(fig)
print(raw_file_counts.to_string())


## Hallazgos parciales

- `INE_CPI` y `MAPA_SLAUGHTER_MAPA` aparecen como fuentes activas con trazabilidad local.
- `MAPA_PRICES_OM` queda trazada como referencia de respaldo y no como feed semanal activo.
- Las fuentes candidatas no deben entrar en el blob oficial ni en el pipeline mixto defendible.


In [ ]:
for source_id in ["INE_CPI", "MAPA_SLAUGHTER_MAPA", "MAPA_PRICES_OM"]:
    subset = source_overview[source_overview["source_id"] == source_id]
    print(f"Source focus: {source_id}")
    if subset.empty:
        print("  source not found")
        continue
    row = subset.iloc[0]
    print(f"  status documented: {row['status']}")
    print(f"  status manifest: {row['evidence_status']}")
    print(f"  official_url: {row['official_url_registry']}")
    print(f"  limitations: {row['limitations_registry']}")


In [ ]:
TABLES.append(save_table(source_overview, "data_sources_audit__mixed_context.csv"))
TABLES.append(save_table(inconsistencies_df, "data_sources_inconsistencies__mixed_context.csv"))
RESULT = {
    "notebook": NOTEBOOK_NAME,
    "tables": TABLES,
    "figures": FIGURES,
    "findings": [
        "Active sources keep official URLs, license URLs and local raw snapshots.",
        "MAPA_PRICES_OM remains traced only and is not treated as active weekly evidence.",
        "Candidate sources remain outside the defended mixed_context route.",
    ],
    "limitations": [
        "The markdown registry is a curated summary and must stay aligned with source manifests.",
    ],
}
print(json.dumps(RESULT, indent=2))


## Limitaciones

Este notebook verifica coherencia documental y presencia local de artefactos. No evalua por si solo la calidad estadistica de las series ni sustituye la inspeccion de raw y procesados.


## Concluson final

La capa de inventario documental queda defendible cuando el estado de cada fuente coincide entre el registro markdown y el manifest JSON, y cuando los raw y derivados exigidos existen localmente.
